# Map synthetic sites and measure nearest-neighbor distances

**Audience:** Python developers preparing reproducible data workflows. **Time:** about 15–25 minutes. **Execution:** offline; no credentials, paid APIs, or downloaded datasets.

Learn to validate geographic coordinates, calculate great-circle distances, and visualize a small site inventory. Distinguish a nearest neighbor by straight-line distance from a route or a service-area assignment.

## Prerequisites and setup

Use the tested Python environment and exact dependency versions in the [collection setup](../../../README.md). Install from the collection root before opening this notebook, then select that environment's kernel. This notebook uses NumPy, pandas and Matplotlib. Restart the kernel and run all cells in order. No working-directory assumptions or private files are required.

## Data provenance

All examples below are newly authored synthetic fixtures. Names, identifiers, coordinates and values are invented for this tutorial and do not represent real people, accounts, organizations or operational records. Code is newly authored for this example; the references explain the underlying APIs.

Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../../AI_DISCLOSURE.md), [license](../../../LICENSE), and [attribution](../../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Build and validate a small coordinate table

A coordinate pair is `(latitude, longitude)` in degrees. Longitude comes second here; many geospatial formats use the reverse order. Explicit column names avoid that ambiguity. Repeated coordinates are legal and produce a zero distance.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sites = pd.DataFrame(
    {
        "site_id": ["site-01", "site-02", "site-03", "site-04"],
        "latitude": [0.0, 0.0, 0.8, 0.9],
        "longitude": [0.0, 0.6, 0.0, 0.9],
    }
)


def validate_coordinates(frame):
    required = {"site_id", "latitude", "longitude"}
    if not required <= set(frame.columns):
        raise ValueError("Missing coordinate columns")
    if frame["site_id"].isna().any() or frame["site_id"].duplicated().any():
        raise ValueError("Site identifiers must be present and unique")
    values = frame[["latitude", "longitude"]].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError("Coordinates must be finite")
    if (np.abs(values[:, 0]) > 90).any() or (np.abs(values[:, 1]) > 180).any():
        raise ValueError("Coordinates outside valid degree ranges")
    return values


coords = validate_coordinates(sites)
sites

## 2. Compute the haversine distance matrix

The haversine intermediate is `a = sin²(Δφ/2) + cos(φ₁)cos(φ₂)sin²(Δλ/2)`, where latitudes `φ` and longitudes `λ` are in radians. The central angle is `δ = 2 × arcsin(sqrt(a))` and distance is `Rδ`, with an approximate mean Earth radius of 6,371 km. Clipping protects against small floating-point overshoots near antipodal points. This is a spherical approximation, not an ellipsoidal survey measurement.

In [ ]:
def haversine_matrix(coordinates):
    angles = np.radians(np.asarray(coordinates, dtype=float))
    lat, lon = angles[:, 0], angles[:, 1]
    dlat = lat[:, None] - lat[None, :]
    dlon = lon[:, None] - lon[None, :]
    a = np.sin(dlat / 2) ** 2 + np.cos(lat[:, None]) * np.cos(lat[None, :]) * np.sin(dlon / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


distances = haversine_matrix(coords)
assert np.allclose(distances, distances.T)
assert np.allclose(np.diag(distances), 0)
assert abs(haversine_matrix([[0, 0], [0, 1]])[0, 1] - 111.195) < 0.01
pd.DataFrame(distances.round(2), index=sites.site_id, columns=sites.site_id)

## 3. Exclude self-matches and summarize

The diagonal would always win an ordinary minimum. Replacing it in a copy preserves the physical distance matrix. This exact all-pairs method uses quadratic memory; use a spatial index for large inventories. Equal-distance ties are resolved by input order.

In [ ]:
candidates = distances.copy()
np.fill_diagonal(candidates, np.inf)
nearest = candidates.argmin(axis=1)
summary = sites.assign(
    neighbor=sites.site_id.to_numpy()[nearest],
    distance_km=candidates[np.arange(len(sites)), nearest].round(2),
)
assert summary.loc[0, "neighbor"] == "site-02"
assert (summary.site_id != summary.neighbor).all()
summary

## 4. Draw a labeled coordinate plot

This deliberately avoids online tiles and geocoding. The equal-angle plot is useful for this tiny equatorial example; it is not a projected global map.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(sites.longitude, sites.latitude, s=70)
for row in sites.itertuples():
    ax.annotate(
        row.site_id, (row.longitude, row.latitude), xytext=(5, 5), textcoords="offset points"
    )
ax.set(
    xlabel="Longitude (degrees)",
    ylabel="Latitude (degrees)",
    title="Invented sites near the equator",
)
ax.set_aspect("equal")
ax.grid(alpha=0.3)
plt.show()
plt.close(fig)

## 5. Exercise rejection and wraparound cases

Validation should reject impossible coordinates before computation. Longitudes across the date line should remain geographically close.

In [ ]:
for invalid in [np.nan, 91.0]:
    bad = sites.copy()
    bad.loc[0, "latitude"] = invalid
    try:
        validate_coordinates(bad)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid latitude accepted")
assert haversine_matrix([[0, 179.9], [0, -179.9]])[0, 1] < 23
assert haversine_matrix([[0, 0], [0, 0]])[0, 1] == 0
print("Coordinate, symmetry, known-distance and boundary checks passed.")

## Validation and interpretation

The assertions above are executable acceptance criteria. A failed assertion is a failed example: inspect the inputs and kernel versions before continuing. Inspect the displayed tables or figures as well; successful execution alone does not establish fitness for a different dataset.

## Limitations and extensions

No addresses, people or geocoding are involved. Distances do not model roads, travel time, accessibility or capacity. Require at least two sites for nearest-neighbor selection and choose an appropriate projection or ellipsoidal method for larger geographic regions.

## Cleanup

This notebook creates no persistent files, network sessions or cloud resources. Restart the kernel to discard in-memory objects. If you adapt it to real data, keep inputs outside a public repository and review every output before sharing.

## References and attribution

- [NumPy broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- [Matplotlib scatter reference](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html)

The example is prepared for editorial review; it is not an IBM endorsement or a claim of production certification.